In [ ]:
import pandas as pd # type: ignore
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Activation, Dense, Dropout, Input, Embedding
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical


In [ ]:
# === Load & Inspect Data ===
url = "https://raw.githubusercontent.com/gyq1985/nlp/main/spam.csv"
df = pd.read_csv(url, encoding='latin-1')[['spamORham', 'Message']]
df = df.rename(columns={'spamORham': 'label', 'Message': 'text'})
display(df.head())
print(df['label'].value_counts())

sns.countplot(x='label', data=df)
plt.title("Class Distribution")
plt.show()


In [ ]:
# === Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# === TF?IDF Vectoriser ===
tfidf = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# Encode labels to numeric
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)


In [ ]:
# === Logistic Regression ===
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)
log_reg.fit(X_train_tfidf, y_train_enc)
y_pred_log = log_reg.predict(X_test_tfidf)

acc_log = accuracy_score(y_test_enc, y_pred_log)
print(f"Logistic Regression Accuracy: {acc_log:.4f}")

print("## Logistic Regression Evaluation")
print(classification_report(y_test_enc, y_pred_log, target_names=le.classes_))

cm_log = confusion_matrix(y_test_enc, y_pred_log)
disp_log = ConfusionMatrixDisplay(confusion_matrix=cm_log, display_labels=le.classes_)
disp_log.plot(cmap='Blues', values_format='d')
plt.title("Logistic Regression ? Confusion Matrix")
plt.show()


In [ ]:
# === Linear SVM (LinearSVC) ===
svm_clf = LinearSVC(class_weight='balanced')
svm_clf.fit(X_train_tfidf, y_train_enc)
y_pred_svm = svm_clf.predict(X_test_tfidf)
acc_log = accuracy_score(y_test_enc, y_pred_log)
print(f"Logistic Regression Accuracy: {acc_log:.4f}")
print("## SVM Evaluation")
print(classification_report(y_test_enc, y_pred_svm, target_names=le.classes_))

cm_svm = confusion_matrix(y_test_enc, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=le.classes_)
disp_svm.plot(cmap='Purples', values_format='d')
plt.title("SVM ? Confusion Matrix")
plt.show()
